# 02 — EDA: Demand Dynamics, Segmentation & Baseline

**Deliverable D2** companion notebook. Loads the Phase 1 Parquet outputs and produces:

1. **Demand dynamics** — Pareto skew, sparsity, monthly seasonality
2. **Revenue & pricing** — category revenue mix
3. **Promotional ROI** — promo transaction share & SKU-week coverage
4. **Inventory health** — snapshot stats (evaluation resource only)
5. **ABC / XYZ segmentation**
6. **Seasonal-naive baseline scorecard** (from `artifacts/metrics/cv_summary.json`)

Governance: `sku_inventory_flags.csv` is never loaded here as a feature source.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from foresight.config import load_settings
from foresight.evaluation.segmentation import abc_xyz_matrix, pareto_summary

settings = load_settings()
processed = settings.processed_dir
print(processed)
print('exists:', processed.exists())

C:\Users\adity\Documents\Zidio\FORESIGHT – AI-Powered Demand & Inventory Intelligence Platform\data\processed
exists: True


In [2]:
weekly = pd.read_parquet(processed / 'sales_weekly.parquet')
calendar = pd.read_parquet(processed / 'calendar.parquet')
promo = pd.read_parquet(processed / 'promo_weekly.parquet')
inventory = pd.read_parquet(settings.interim_dir / 'inventory_normalized.parquet')
sku = pd.read_csv(settings.dataset_file('raw_sku'), dtype={'sku_id': str})

print(f'weekly: {weekly.shape} | skus: {weekly.sku_id.nunique()} | weeks: {weekly.year_week.nunique()}')
weekly.head(3)

weekly: (1050000, 7) | skus: 5000 | weeks: 210


,sku_id,year_week,units_sold,revenue,gross_revenue,unit_price,promo_trans_count
0,SKU00001,2021-W52,3,2440.229980,2440.229980,813.409973,0
1,SKU00001,2022-W01,14,11387.739258,11387.739258,813.409973,0
2,SKU00001,2022-W02,29,23588.890625,23588.888672,813.409973,0


## 1. Demand Dynamics

### Pareto curve — share of revenue vs share of SKUs

In [3]:
rev = weekly.groupby('sku_id')['revenue'].sum().sort_values(ascending=False)
cum = rev.cumsum() / rev.sum()

p = pareto_summary(weekly)
print(p)

fig = go.Figure()
fig.add_trace(go.Scatter(y=cum.values, mode='lines', name='Cumulative revenue share'))
fig.add_hline(y=0.70, line_dash='dash', annotation_text='70% revenue')
fig.add_vline(x=p['top_sku_count'], line_dash='dash', annotation_text='Top 20% SKUs')
fig.update_layout(title='Pareto curve (80/21 rule)', xaxis_title='SKUs (ranked)',
                  yaxis_title='Cumulative revenue share', template='plotly_white')
fig.show()

{'n_skus': 5000, 'top_sku_count': 1000, 'top_revenue_share': 0.7307554483413696}


In [4]:
wk_month = calendar[['year_week', 'month']].drop_duplicates('year_week')
m = weekly.merge(wk_month, on='year_week', how='left')
monthly = m.groupby('month')['units_sold'].sum().sort_index()
avg = monthly.mean()

fig = px.bar(x=monthly.index, y=monthly.values,
             labels={'x': 'ISO Month', 'y': 'Units'},
             title='Monthly demand seasonality')
fig.add_hline(y=avg, line_dash='dash', annotation_text='average')
fig.update_layout(template='plotly_white')
fig.show()

zero_share = (weekly['units_sold'] == 0).mean()
print(f'Zero-demand SKU-weeks: {zero_share:.2%}')

Zero-demand SKU-weeks: 1.61%


## 3. Promotional Activity

Precedence: highest discount wins across the 41 overlapping campaign pairs.

In [5]:
n_promo_txn = int(weekly['promo_trans_count'].sum())
n_total_txn = int(settings.audit_expectations.n_sales_contaminated)
print(f'Promo share of transactions: {n_promo_txn / n_total_txn:.2%}')

active = weekly[(weekly['promo_trans_count'] > 0) & (weekly['units_sold'] > 0)]
print(f'Units in promo SKU-weeks: {active["units_sold"].sum() / weekly["units_sold"].sum():.2%}')

promo_by_week = weekly.groupby('year_week', observed=True)['promo_trans_count'].sum()
fig = px.line(promo_by_week, title='Promo transactions per week')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

Promo share of transactions: 21.02%
Units in promo SKU-weeks: 27.28%


## 4. Inventory Health

Snapshot statistics only — `sku_inventory_flags.csv` remains an out-of-sample evaluation resource.

In [6]:
print(f'records: {len(inventory):,}')
print(f'mean stock: {inventory["stock_on_hand"].mean():.2f} | median: {inventory["stock_on_hand"].median():.1f}')
print(f'outages (stock <= 0): {(inventory["stock_on_hand"] <= 0).sum():,}')

lead = inventory.groupby('category')['lead_time_days'].first().sort_values()
fig = px.bar(x=lead.values, y=lead.index, orientation='h',
             labels={'x': 'Lead time (days)', 'y': 'Category'},
             title='Category lead times (configs/risk.yaml)')
fig.update_layout(template='plotly_white')
fig.show()

records: 26,408
mean stock: 166.20 | median: 140.0
outages (stock <= 0): 4,239


## 5. ABC / XYZ Segmentation

In [7]:
seg = abc_xyz_matrix(weekly, sku)
print(seg['abc'].value_counts().sort_index())
print(seg['xyz'].value_counts().sort_index())

ct = pd.crosstab(seg['abc'], seg['xyz'])
fig = px.imshow(ct, text_auto=True, title='ABC × XYZ matrix (SKU count)')
fig.update_layout(template='plotly_white')
fig.show()

if 'category' in seg.columns:
    mix = seg.groupby('category')['total_revenue'].sum().sort_values(ascending=False)
    fig = px.bar(x=mix.values, y=mix.index, orientation='h',
                 labels={'x': 'Revenue (PKR)', 'y': 'Category'},
                 title='Revenue by category')
    fig.update_layout(template='plotly_white')
    fig.show()

abc
A     873
B    1348
C    2779
Name: count, dtype: int64
xyz
X    1911
Y    3089
Name: count, dtype: int64


## 6. Seasonal-Naive Baseline Scorecard

Mandatory benchmark (Rule 4): ML models must beat these WAPE numbers on identical folds.

In [8]:
summary = json.loads((settings.artifacts_dir / 'metrics' / 'cv_summary.json').read_text())
display(pd.DataFrame(summary['folds']))
print(f"Overall pooled WAPE: {summary['overall_wape']:.2%}")
print(f"Overall bias: {summary['overall_bias']:+.4f}")

,fold,train_end_week,valid_weeks,n_rows,wape,bias,mae,rmse
0,1,2024-W52,8,40000,0.380051,0.062236,6.245456,11.692543
1,2,2025-W16,8,40000,0.352940,0.043147,6.630525,10.685014
2,3,2025-W40,12,60000,0.390487,-0.025923,8.491412,37.327771


Overall pooled WAPE: 37.76%
Overall bias: +0.0146
